In [1]:
from datasets import load_dataset
ds = load_dataset("timm/oxford-iiit-pet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/378M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/413M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3680 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3669 [00:00<?, ? examples/s]

In [2]:
# Inspect dataset features
print(f"Dataset features: {ds['train'].features}")
print(f"Example item keys: {ds['train'][0].keys()}")

Dataset features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'bengal', 'birman', 'bombay', 'boxer', 'british_shorthair', 'chihuahua', 'egyptian_mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'maine_coon', 'miniature_pinscher', 'newfoundland', 'persian', 'pomeranian', 'pug', 'ragdoll', 'russian_blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'siamese', 'sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']), 'image_id': Value('string'), 'label_cat_dog': ClassLabel(names=['cat', 'dog'])}
Example item keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])


In [5]:
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Data processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# PyTorch and deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report
)

# Formatting
from tabulate import tabulate

# ==================== DEVICE AND GPU SETUP ====================
print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# GPU optimizations (for T4 or similar)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

# ==================== RANDOM SEED FOR REPRODUCIBILITY ====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch: 2.10.0+cpu
device: cpu


In [6]:
import torchvision.transforms as T

# Inspect features to be sure
print(f"Available keys: {ds['train'].features.keys()}")

class PetSegmentationDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
        # Dynamically find the mask key
        self.mask_key = 'mask' if 'mask' in hf_dataset.features else 'segmentation_mask'

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert("RGB")
        mask = item[self.mask_key]

        if self.transform:
            image = self.transform(image)
            mask = T.Resize((128, 128), interpolation=T.InterpolationMode.NEAREST)(mask)
            mask = torch.from_numpy(np.array(mask)).long()
            # Adjust mask labels: 1: Foreground, 2: Background, 3: Outline
            # Map to 0, 1, 2 for CrossEntropyLoss
            mask = mask - 1

        return image, mask

# Define transforms
image_transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Prepare data loaders
train_ds = PetSegmentationDataset(ds['train'], transform=image_transform)
test_ds = PetSegmentationDataset(ds['test'], transform=image_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Dataset initialized. Using mask key: {train_ds.mask_key}")

Available keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])
Dataset initialized. Using mask key: segmentation_mask
